# 03 · Train the binding-affinity ranker

Regresses $\mathrm{pK} = -\log_{10} K_d/K_i$ on the protein–ligand interaction graph. This is
the model that **ranks** generated molecules in notebook 04.

**Input** → `graphs_affinity/` from notebook 01 §8 (PDBBind / PDBBind+ labels).
**Output** → `affinity_best.pt`.

### Why a separate network from the generator
Different data and different inductive bias. The generator trains on ~10⁵ unlabelled BioLiP
complexes and only needs plausible chemistry in a cavity. Ranking needs the ~2·10⁴ labelled
PDBBind complexes and a scalar output, so the right symmetry is **invariance**, not
equivariance — the same `EGCL` layer with coordinate updates switched off, since it consumes
only interatomic distances.

Four edge families carry different physics and get separate projections: covalent ligand bonds,
through-space ligand contacts, pocket contacts, and ligand–pocket contacts (the interaction
term the label actually depends on).

### Read the numbers honestly
A CASF-2016 core-set Pearson *r* of 0.7–0.8 is where good published scoring functions sit.
Much above that on a random split usually means leakage, which is why notebook 01 splits by
PDB entry and forces the core set into test.

In [ ]:
# --- locate the molgen package ---------------------------------------------
# On Kaggle, make `molgen` importable in one of these ways:
#   1. Add the "molgen-src" Dataset (upload this repo's `src/` folder), or
#   2. Add the output of notebook 01 (it ships a copy of `src/molgen`), or
#   3. Run the notebook from a clone of the repo.
import glob
import subprocess
import sys
from pathlib import Path


def find_molgen():
    patterns = [
        "/kaggle/input/*/src", "/kaggle/input/*", "/kaggle/input/*/*",
        "/kaggle/working/src", "./src", "../src", ".", "..",
    ]
    for pattern in patterns:
        for candidate in sorted(glob.glob(pattern)):
            if (Path(candidate) / "molgen" / "__init__.py").is_file():
                return str(Path(candidate).resolve())
    return None


MOLGEN_SRC = find_molgen()
if MOLGEN_SRC is None:
    raise FileNotFoundError(
        "Could not find the `molgen` package.\n"
        "Attach the molgen-src dataset (the repo's src/ folder, containing molgen/), "
        "or the output of notebook 01, then re-run this cell."
    )
if MOLGEN_SRC not in sys.path:
    sys.path.insert(0, MOLGEN_SRC)
print("molgen source:", MOLGEN_SRC)

In [ ]:
# --- dependencies -----------------------------------------------------------
# torch_geometric is required. We deliberately avoid torch_scatter/torch_cluster
# (they need wheels matched to the exact torch build and are the usual cause of
# Kaggle install failures) -- molgen uses scipy KD-trees and native torch ops.
# rdkit is optional: without it you lose SMILES/QED/SA but nothing else.
def ensure(module: str, package: str | None = None) -> bool:
    try:
        __import__(module)
        return True
    except ImportError:
        pkg = package or module
        print(f"installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
    try:
        __import__(module)
        return True
    except ImportError:
        print(f"  !! {module} unavailable")
        return False


HAS_PYG = ensure("torch_geometric")
HAS_RDKIT = ensure("rdkit")
assert HAS_PYG, "torch_geometric is required"
print("rdkit:", HAS_RDKIT)

In [ ]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import molgen
from molgen import (
    affinity, chem, dataset, diffusion, egnn, evaluate, graphs, pdbio, pockets, sources,
)
from molgen.train_utils import (
    EMA, History, TimeBudget, count_parameters, device_info, grad_norm,
    load_checkpoint, save_checkpoint, set_seed, warmup_cosine_lambda,
)

warnings.filterwarnings("ignore", category=UserWarning)
set_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("molgen", molgen.__version__, "| device:", device)
print(device_info())

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")
WORK.mkdir(parents=True, exist_ok=True)
print("working dir:", WORK)

In [ ]:
AFF_DIR = next(
    (p for p in [Path(g) for g in glob.glob("/kaggle/input/*/graphs_affinity")] + [WORK / "graphs_affinity"]
     if (p / "train_manifest.json").exists()),
    None,
)
assert AFF_DIR is not None, (
    "No graphs_affinity/ found. Run notebook 01 section 8 with PDBBind attached — "
    "the BioLiP set has no affinity labels."
)
print("affinity graphs:", AFF_DIR)

# These sets are small enough to hold in RAM, which makes the epochs much faster.
train_list = dataset.load_all(AFF_DIR, "train")
val_list = dataset.load_all(AFF_DIR, "val")
test_list = dataset.load_all(AFF_DIR, "test")


def labelled(items):
    return [d for d in items if torch.isfinite(d.y).all()]


train_list, val_list, test_list = labelled(train_list), labelled(val_list), labelled(test_list)
y_train = np.array([float(d.y) for d in train_list])
print(f"train {len(train_list):,} | val {len(val_list):,} | test {len(test_list):,}")
print(f"pK: mean {y_train.mean():.2f}  std {y_train.std():.2f}  range [{y_train.min():.2f}, {y_train.max():.2f}]")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(y_train, bins=50, color="#3b82f6"); ax[0].set_title("train pK"); ax[0].set_xlabel("pK")
ax[1].scatter([d.lig_pos.shape[0] for d in train_list], y_train, s=3, alpha=0.25)
ax[1].set_xlabel("ligand heavy atoms"); ax[1].set_ylabel("pK")
ax[1].set_title("affinity vs ligand size (the trivial baseline)")
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

from scipy.stats import pearsonr
r_size = pearsonr([d.lig_pos.shape[0] for d in train_list], y_train)[0]
print(f"size-only Pearson r = {r_size:.3f}  <- the model must clearly beat this")

## Configuration

Labels are standardised by the training mean/std inside the model, which is worth a few epochs
of convergence on its own.

In [ ]:
ACFG = affinity.AffinityConfig(
    hidden=128,
    num_layers=4,
    edge_embed_dim=32,
    ligand_contact_cutoff=5.0,
    rbf_cutoff=8.0,
    dropout=0.1,
    label_mean=float(y_train.mean()),
    label_std=float(y_train.std()),
)

ATRAIN = dict(
    epochs=150, batch_size=24, lr=3e-4, weight_decay=1e-6,
    grad_clip=5.0, warmup_steps=300, huber_delta=1.0,
    patience=25, num_workers=2, session_hours=8.5,
)

ABEST = WORK / "affinity_best.pt"
ALAST = WORK / "affinity_last.pt"
print(ACFG)

In [ ]:
amodel = affinity.AffinityNet(ACFG).to(device)
print(f"{count_parameters(amodel):,} trainable parameters")

atrain_loader = dataset.make_loader(train_list, batch_size=ATRAIN["batch_size"], shuffle=True,
                                    num_workers=ATRAIN["num_workers"], drop_last=True)
aval_loader = dataset.make_loader(val_list, batch_size=ATRAIN["batch_size"], shuffle=False,
                                  num_workers=ATRAIN["num_workers"])

aopt = torch.optim.AdamW(amodel.parameters(), lr=ATRAIN["lr"], weight_decay=ATRAIN["weight_decay"])
asched = torch.optim.lr_scheduler.LambdaLR(
    aopt, warmup_cosine_lambda(ATRAIN["warmup_steps"], ATRAIN["epochs"] * max(len(atrain_loader), 1))
)
aema = EMA(amodel, decay=0.995)
ahistory = History(WORK / "affinity_history.jsonl")
abudget = TimeBudget(limit_s=ATRAIN["session_hours"] * 3600)


@torch.no_grad()
def evaluate_affinity(net, items, batch_size=32):
    pred = affinity.predict_batched(net, items, device=str(device), batch_size=batch_size).numpy()
    target = np.array([float(d.y) for d in items])
    return affinity.regression_metrics(pred, target), pred, target

## Train

In [ ]:
best_rmse, bad_epochs = float("inf"), 0
for epoch in range(ATRAIN["epochs"]):
    amodel.train()
    total, nb = 0.0, 0
    for batch in atrain_loader:
        out = amodel.loss(batch.to(device), huber_delta=ATRAIN["huber_delta"])
        aopt.zero_grad(set_to_none=True)
        out["loss"].backward()
        torch.nn.utils.clip_grad_norm_(amodel.parameters(), ATRAIN["grad_clip"])
        aopt.step(); asched.step(); aema.update(amodel)
        total += float(out["loss"]); nb += 1

    aema.apply_to(amodel)
    vm, _, _ = evaluate_affinity(amodel, val_list)
    aema.restore(amodel)
    ahistory.log(step=epoch, epoch=epoch, train_loss=total / max(nb, 1), **{f"val_{k}": v for k, v in vm.items()})

    flag = ""
    if vm["rmse"] < best_rmse:
        best_rmse, bad_epochs = vm["rmse"], 0
        save_checkpoint(ABEST, amodel, aopt, asched, aema, epoch, epoch,
                        config=amodel.config_dict(), extra={"val": vm})
        flag = "  <- best"
    else:
        bad_epochs += 1
    save_checkpoint(ALAST, amodel, aopt, asched, aema, epoch, epoch, config=amodel.config_dict())

    if epoch % 5 == 0 or flag:
        print(f"e{epoch:>3} train {total/max(nb,1):.4f} | val RMSE {vm['rmse']:.3f} "
              f"MAE {vm['mae']:.3f} r {vm['pearson']:.3f} rho {vm['spearman']:.3f}{flag}", flush=True)

    if bad_epochs >= ATRAIN["patience"]:
        print(f"\nearly stop: no val improvement for {ATRAIN['patience']} epochs")
        break
    if abudget.expired():
        print(f"\ntime budget reached ({abudget})")
        break

print(f"\nbest val RMSE {best_rmse:.3f}")

In [ ]:
adf = ahistory.to_frame()
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
ax[0].plot(adf.epoch, adf.train_loss, label="train (huber)")
ax[0].plot(adf.epoch, adf.val_rmse, label="val RMSE"); ax[0].set_title("loss")
ax[1].plot(adf.epoch, adf.val_pearson, label="Pearson r")
ax[1].plot(adf.epoch, adf.val_spearman, label="Spearman rho")
ax[1].axhline(r_size, color="crimson", ls="--", label="size-only baseline")
ax[1].set_title("val correlation")
ax[2].plot(adf.epoch, adf.val_mae); ax[2].set_title("val MAE (pK units)")
for a in ax: a.set_xlabel("epoch"); a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Test-set evaluation

`sd` is the standard deviation of the residuals, reported alongside RMSE in the PDBBind/CASF
literature. If the CASF-2016 core set was available it is inside this test split.

In [ ]:
load_checkpoint(ABEST, amodel, map_location=device)
amodel = amodel.to(device)

for name, items in [("val", val_list), ("test", test_list)]:
    m, pred, target = evaluate_affinity(amodel, items)
    print(f"{name:>5}: " + "  ".join(f"{k}={v:.3f}" if isinstance(v, float) else f"{k}={v}"
                                     for k, v in m.items()))
    if name == "test":
        test_pred, test_target = pred, target

fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))
lim = [min(test_target.min(), test_pred.min()) - 0.5, max(test_target.max(), test_pred.max()) + 0.5]
ax[0].scatter(test_target, test_pred, s=10, alpha=0.4)
ax[0].plot(lim, lim, "k--", lw=1)
ax[0].set_xlabel("experimental pK"); ax[0].set_ylabel("predicted pK")
mt = affinity.regression_metrics(test_pred, test_target)
ax[0].set_title(f"test: r={mt['pearson']:.3f}  RMSE={mt['rmse']:.2f}")
ax[0].set_xlim(lim); ax[0].set_ylim(lim)
ax[1].hist(test_pred - test_target, bins=40, color="#f59e0b")
ax[1].set_xlabel("predicted - experimental"); ax[1].set_title("residuals")
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

(WORK / "affinity_test_metrics.json").write_text(json.dumps(mt, indent=2))
print(json.dumps(mt, indent=2))

---
**Next:** `04_generate_and_rank.ipynb` — attach this notebook's output together with
notebook 02's for the full pipeline.